<a href="https://colab.research.google.com/github/Cavalheiro93/mvp-machine-learning-spotify-tracks/blob/main/projeto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Machine Learning | Spotify Tracks

## 00. Sumário & Setup

| **Recursos de Áudio**   | **Descrição** |
|--------------------------|---------------|
| ``track_id``                 | Id ou código da faixa |
| ``artists``                  | Nome do artista ou banda da faixa |
| ``album_name``               | Nome do album da faixa |
| ``track_name``               | Nome da faixa musical |
| ``popularidade``             | Popularidade da faixa (0 a 100) |
| ``duration_ms``              | Duração em milisegundos da faixa |
| ``explicit``                 | Binário, se a faixa é explicita ou não (se contém palavrão ou não) |
| ``danceability``             | Adequação da faixa para dança (0,0 a 1,0) |
| ``energy``                   | Medida perceptiva de intensidade e atividade (0,0 a 1,0) |
| ``key``                      | Tom da faixa, varia de -1 a -11 |
| ``loudness``                 | Volume geral da faixa em decibéis (-60 a 0 dB) |
| ``mode``                     | Modalidade da faixa (Maior ‘1’ / Menor ‘0’) |
| ``speechiness``              | Presença de palavras faladas na faixa |
| ``acousticness``             | Medida de confiança (0 a 1) de que a faixa é acústica |
| ``instrumentalness``         | Indica se a faixa contém vocais (0,0 a 1,0) |
| ``liveness``                 | Presença de público na gravação (0,0 a 1,0) |
| ``valence``                  | Positividade musical (0,0 a 1,0) |
| ``tempo``                    | Tempo da faixa em batidas por minuto (BPM), Na terminologia musical indica a velocidade ou ritmo da música. |
| ``time_signature``           | Compasso estimado (3 a 7) |
| ``track_genre``              | O gênero ao qual a faixa pertence |

## 01. Contexto & Definição do Problema

Criar um modelo que recomende músicas de rock de acordo com o humor/atividade do usuário (ex.: academia, leitura, relax).

Objetivo prático: dado um conjunto de atributos musicais, classificar cada faixa em um rótulo de humor/atividade

## 02. Dados & Carregamento (via URL do GitHub)

### Imports

In [1]:
# configuração para não exibir os warnings
import warnings
warnings.filterwarnings("ignore")

import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier

In [2]:

pd.set_option('display.max_columns', None)

# Filtro das colunas que usaremos somente
usecols = ['artist_name', 'track_name', 'track_id', 'popularity', 'genre', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# O estudo será baseado apenas em gêneros de rock
genre_rock = ['alt-rock', 'black-metal', 'death-metal', 'emo', 'hard-rock', 'hardcore', 'heavy-metal', 'metal', 'metalcore',
              'goth', 'psych-rock', 'punk', 'punk-rock', 'rock', 'rock-n-roll', 'grunge', 'alternative']

df = pd.read_csv("hf://datasets/maharshipandya/spotify-tracks-dataset/dataset.csv", low_memory=False)

#renomear track_genre para genre
df = df.rename(columns={'track_genre': 'genre', 'artists': 'artist_name'})

df = df[usecols]

# Somente generos de rock
filtro_rock = df['genre'].isin(genre_rock)
#Somente musicas com popularidade maior que 30, para eliminar musicas muito desconhecidas
filtro_popularidade = df['popularity'] > 1
# Somente musicas com instrumentalidade maior que 0.5, para eliminar podcasts
#filtro_instrumentalness = df['instrumentalness'] > 0.1
#speechiness maior que 0
filtro_speechiness = df['speechiness'] != 0

# Aplicando os filtros
df = df[filtro_rock]
#df = df[filtro_instrumentalness]
df = df[filtro_popularidade]
df = df[filtro_speechiness]
